# Refinement Model Debugging Notebook

This notebook provides a step-by-step debugging environment for the protein structure refinement model. It allows testing individual components of the training pipeline:

1. Dataset creation and loading
2. Dataloader setup with collation
3. Model initialization
4. Forward pass through the model
5. Loss calculation
6. Gradient backpropagation
7. Parameter updates

Each section can be run independently to debug specific parts of the pipeline.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import pickle
from typing import Dict, List, Optional, Tuple, Any

# Import the dataset and model classes
from openfold.model.mmc.data import (
    ProteinRefinementDataset, 
    get_dataloader,
    ProteinDataCollator
)
from openfold.model.mmc.model import MMCRefinementModel, backprop_energy_gradient
from openfold.model.mmc.loss import RefinementLoss
from openfold.model.structure_module import StructureModule
from openfold.utils.tensor_utils import tensor_tree_map

[2025-04-21 22:15:35,674] [INFO] [real_accelerator.py:161:get_accelerator] Setting ds_accelerator to cuda (auto detect)


## 1. Dataset Creation and Loading

First, we'll set up the dataset with a small subset of proteins, similar to the test_refinement_dataset.ipynb approach.

In [5]:
# Path to the test predictions directory
project_root = "/gpfs/data/rsingh47/hp_protein_folding/protein_folding"
predictions_dir = os.path.join(project_root, "output_mmc", "test_predictions")
print(f"Predictions directory: {predictions_dir}")
print(f"Directory exists: {os.path.exists(predictions_dir)}")

# List the protein directories in the test predictions
protein_dirs = [d for d in os.listdir(predictions_dir) 
               if os.path.isdir(os.path.join(predictions_dir, d))]
print(f"Found {len(protein_dirs)} protein directories")
print(f"First 5 proteins: {protein_dirs[:5]}")

Predictions directory: /gpfs/data/rsingh47/hp_protein_folding/protein_folding/output_mmc/test_predictions
Directory exists: True
Found 22 protein directories
First 5 proteins: ['1C07_A', '1C1K_A', '1C39_A', '1C3P_A', '1C52_A']


In [6]:
# Select a small subset of proteins for testing
test_proteins = protein_dirs[:5]  # Just use the first 3 proteins
print(f"Using proteins: {test_proteins}")

# Create the dataset
dataset = ProteinRefinementDataset(
    predictions_dir=predictions_dir,
    pH='5',  # Use pH 5 for ground truth selection
    filter_proteins=test_proteins,
    max_samples_per_protein=2,  # Limit samples per protein for testing
    cache_embeddings=True  # Cache embeddings for faster access
)

print(f"Dataset created with {len(dataset)} samples")

Using proteins: ['1C07_A', '1C1K_A', '1C39_A', '1C3P_A', '1C52_A']
Dataset created with 10 samples


In [5]:
print(dataset[0].keys())

dict_keys(['protein_name', 'gt_atom37_positions', 'gt_atom37_mask', 'intermediate_checkpoint_forces', 'checkpoint_number', 'embeddings', 'sequence_length'])


In [7]:
print(dataset[0]['embeddings'].keys())
print('embeddings')
for key, value in dataset[0]['embeddings']['s_inputs'].items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: Tensor with shape {value.shape} and dtype {value.dtype}")
    elif isinstance(value, dict):
        print(f"  {key}: dict with keys {list(value.keys())}")
    else:
        print(f"  {key}: {type(value)}")

print('feats')
for key, value in dataset[0]['embeddings']['feats'].items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: Tensor with shape {value.shape} and dtype {value.dtype}")
    elif isinstance(value, dict):
        print(f"  {key}: dict with keys {list(value.keys())}")
    else:
        print(f"  {key}: {type(value)}")

dict_keys(['s_inputs', 'feats'])
embeddings
  msa: Tensor with shape torch.Size([512, 95, 256]) and dtype torch.float32
  pair: Tensor with shape torch.Size([95, 95, 128]) and dtype torch.float32
  single: Tensor with shape torch.Size([95, 384]) and dtype torch.float32
feats
  aatype: Tensor with shape torch.Size([95]) and dtype torch.int64
  residue_index: Tensor with shape torch.Size([95]) and dtype torch.int64
  seq_length: Tensor with shape torch.Size([]) and dtype torch.int64
  seq_mask: Tensor with shape torch.Size([95]) and dtype torch.float32
  msa_mask: Tensor with shape torch.Size([512, 95]) and dtype torch.float32
  msa_row_mask: Tensor with shape torch.Size([512]) and dtype torch.float32
  atom14_atom_exists: Tensor with shape torch.Size([95, 14]) and dtype torch.float32
  residx_atom14_to_atom37: Tensor with shape torch.Size([95, 14]) and dtype torch.int64
  residx_atom37_to_atom14: Tensor with shape torch.Size([95, 37]) and dtype torch.int64
  atom37_atom_exists: Tensor w

In [57]:
print(dataset[0]['embeddings']['feats'].keys())

dict_keys(['aatype', 'residue_index', 'seq_length', 'seq_mask', 'msa_mask', 'msa_row_mask', 'atom14_atom_exists', 'residx_atom14_to_atom37', 'residx_atom37_to_atom14', 'atom37_atom_exists', 'extra_msa', 'extra_msa_mask', 'extra_msa_row_mask', 'bert_mask', 'true_msa', 'extra_has_deletion', 'extra_deletion_value', 'msa_feat', 'target_feat', 'use_clamped_fape'])


## 3. DataLoader Setup with Collation

Now let's set up the DataLoader with the appropriate collation function to handle batching.

In [8]:
# Define feature keys to extract
feature_keys = ["pair", "single", "ground_truth_atom_positions", "aatype", "residue_index"]

# Create a data collator
collator = ProteinDataCollator(feature_keys=feature_keys)

# Create a DataLoader
batch_size = min(3, len(dataset))  # Use a small batch size for testing
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=batch_size,
    collate_fn=collator,
    shuffle=False
)

print(f"Created DataLoader with batch size {batch_size}")

Created DataLoader with batch size 3


In [9]:
# Define feature keys to extract
feature_keys = ["pair", "single", "ground_truth_atom_positions", "aatype", "residue_index"]

# Create a data collator
uncropped_collator = ProteinDataCollator(feature_keys=feature_keys, crop=None)

# Create a DataLoader
batch_size = min(3, len(dataset))  # Use a small batch size for testing
uncropped_dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=batch_size,
    collate_fn=uncropped_collator,
    shuffle=False
)

print(f"Created DataLoader with batch size {batch_size}")

Created DataLoader with batch size 3


In [10]:
# Load a batch and examine its structure
print("\nLoading a batch...")
batch = next(iter(dataloader))

print("\nBatch keys:")
for key, value in batch.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: Tensor with shape {value.shape} and dtype {value.dtype}")
    elif isinstance(value, dict):
        print(f"  {key}: dict with keys {list(value.keys())}")
    else:
        print(f"  {key}: {type(value)}")


Loading a batch...

Batch keys:
  pair: Tensor with shape torch.Size([3, 217, 217, 128]) and dtype torch.float32
  single: Tensor with shape torch.Size([3, 217, 384]) and dtype torch.float32
  pair_mask: Tensor with shape torch.Size([3, 217, 217]) and dtype torch.float32
  seq_mask: Tensor with shape torch.Size([3, 217]) and dtype torch.float32
  all_atom_positions: Tensor with shape torch.Size([3, 217, 37, 3]) and dtype torch.float64
  all_atom_mask: Tensor with shape torch.Size([3, 217, 37]) and dtype torch.float64
  aatype: Tensor with shape torch.Size([3, 217]) and dtype torch.int64
  atom14_atom_exists: Tensor with shape torch.Size([3, 217, 14]) and dtype torch.float32
  residx_atom14_to_atom37: Tensor with shape torch.Size([3, 217, 14]) and dtype torch.int64
  residx_atom37_to_atom14: Tensor with shape torch.Size([3, 217, 37]) and dtype torch.int64
  atom37_atom_exists: Tensor with shape torch.Size([3, 217, 37]) and dtype torch.float32
  atom14_gt_exists: Tensor with shape torch

In [60]:
data = None
for i, batch in enumerate(dataloader):
    print(batch['pair'].size())
    if i == 2:
        data = batch

uncropped_data = None
for i, batch in enumerate(uncropped_dataloader):
    print(batch['pair'].size())
    if i == 2:
        uncropped_data = batch

torch.Size([3, 217, 217, 128])
torch.Size([3, 217, 217, 128])
torch.Size([3, 256, 256, 128])
torch.Size([1, 131, 131, 128])
torch.Size([3, 217, 217, 128])
torch.Size([3, 217, 217, 128])
torch.Size([3, 375, 375, 128])
torch.Size([1, 131, 131, 128])


In [62]:
data['feats']["atom14_atom_exists"][2,150,:]
data['feats']['aatype']

tensor([[19, 14, 11,  7,  0,  1,  6, 11, 18,  2,  9,  7,  7, 18,  6,  2, 14, 19,
         15, 18,  0, 12, 13, 16,  7, 15, 15, 10,  0, 16,  7, 15, 16, 19,  5,  0,
          9,  6,  6, 13, 10, 11,  7,  2, 19,  0, 13,  2, 14,  0,  7,  7, 12,  8,
          8,  0, 13, 11, 15,  1,  0,  2,  7, 13,  4, 18,  9,  2,  2, 14,  0, 19,
          7,  9,  6, 18, 10,  1, 11, 11,  7, 13, 11,  1,  9, 10, 18,  9,  3, 10,
          3,  0,  8,  8,  4,  3,  7, 19,  5,  6,  0, 13, 18,  3, 16,  3,  5, 19,
         13, 19, 10, 15, 10,  8,  5, 15, 14,  6, 18,  0, 13, 14, 13,  6, 11,  7,
         13, 10,  6,  6,  9,  7,  6,  7, 11,  7, 11,  7, 18,  2, 10,  2,  9, 14,
         10, 14, 11,  7, 10,  2,  3,  2,  6, 13, 10, 13,  0, 10,  6, 11, 15, 10,
          6,  9, 19, 11,  6, 19, 13,  6, 14,  6, 19, 18, 10, 10,  5, 10,  7, 16,
          3, 14, 10, 10,  6,  3, 18, 10, 15, 11, 13,  2, 10, 15,  2, 19,  0, 13,
         10, 11,  0, 13,  2,  9, 19,  1,  6, 19, 13,  7,  6,  7, 19, 18, 10,  7,
          7,  7,  7, 18,  8,

In [63]:
device = torch.device("cuda" if torch.cuda.is_available else "cpu")
feats = data["feats"]
outs = sm({"pair":data["pair"].to(device), "single":data["single"].to(device)},
                feats["aatype"].to(device) if "aatype" in feats else None,
                mask=feats["seq_mask"].to(device),
                inplace_safe=True)
cropped_pos = outs["positions"][-1]

In [64]:
from openfold.model.mmc.metrics import calculate_ca_rmsd

calculate_ca_rmsd(cropped_pos, data["atom14_gt_positions"].to(device), data["atom14_atom_exists"].to(device))

torch.Size([3, 256, 14, 3])
torch.Size([3, 256, 14, 3])


tensor([1.9438, 3.8764, 1.4232], device='cuda:0', dtype=torch.float64)

In [65]:
data["aatype"]

tensor([[19, 14, 11,  7,  0,  1,  6, 11, 18,  2,  9,  7,  7, 18,  6,  2, 14, 19,
         15, 18,  0, 12, 13, 16,  7, 15, 15, 10,  0, 16,  7, 15, 16, 19,  5,  0,
          9,  6,  6, 13, 10, 11,  7,  2, 19,  0, 13,  2, 14,  0,  7,  7, 12,  8,
          8,  0, 13, 11, 15,  1,  0,  2,  7, 13,  4, 18,  9,  2,  2, 14,  0, 19,
          7,  9,  6, 18, 10,  1, 11, 11,  7, 13, 11,  1,  9, 10, 18,  9,  3, 10,
          3,  0,  8,  8,  4,  3,  7, 19,  5,  6,  0, 13, 18,  3, 16,  3,  5, 19,
         13, 19, 10, 15, 10,  8,  5, 15, 14,  6, 18,  0, 13, 14, 13,  6, 11,  7,
         13, 10,  6,  6,  9,  7,  6,  7, 11,  7, 11,  7, 18,  2, 10,  2,  9, 14,
         10, 14, 11,  7, 10,  2,  3,  2,  6, 13, 10, 13,  0, 10,  6, 11, 15, 10,
          6,  9, 19, 11,  6, 19, 13,  6, 14,  6, 19, 18, 10, 10,  5, 10,  7, 16,
          3, 14, 10, 10,  6,  3, 18, 10, 15, 11, 13,  2, 10, 15,  2, 19,  0, 13,
         10, 11,  0, 13,  2,  9, 19,  1,  6, 19, 13,  7,  6,  7, 19, 18, 10,  7,
          7,  7,  7, 18,  8,

In [66]:
device = torch.device("cuda" if torch.cuda.is_available else "cpu")
feats = uncropped_data["feats"]
outs = sm({"pair":uncropped_data["pair"].to(device), "single":uncropped_data["single"].to(device)},
                feats["aatype"].to(device) if "aatype" in feats else None,
                mask=feats["seq_mask"].to(device),
                inplace_safe=True)
uncropped_pos = outs["positions"][-1]

In [67]:
from openfold.model.mmc.metrics import calculate_ca_rmsd

calculate_ca_rmsd(uncropped_pos, uncropped_data["atom14_gt_positions"].to(device), uncropped_data["atom14_atom_exists"].to(device))

torch.Size([3, 375, 14, 3])
torch.Size([3, 375, 14, 3])


tensor([1.4826, 4.8054, 1.4232], device='cuda:0', dtype=torch.float64)

In [35]:
calculate_ca_rmsd(uncropped_data["atom14_gt_positions"].to(device), uncropped_data["atom14_gt_positions"].to(device), uncropped_data["atom14_atom_exists"].to(device))

torch.Size([3, 375, 14, 3])
torch.Size([3, 375, 14, 3])


tensor([1.2892e-06, 3.0266e-06, 1.6004e-06], device='cuda:0',
       dtype=torch.float64)

In [47]:
uncropped_pos[2, 374, :]

tensor([[-12.2494, -14.7973,  10.9757],
        [-13.6676, -14.4792,  11.1199],
        [-13.9009, -12.9724,  11.0579],
        [-14.8445, -12.5101,  10.4122],
        [-14.2093, -15.0449,  12.4304],
        [ -0.0000,  -0.0000,   0.0000],
        [ -0.0000,  -0.0000,   0.0000],
        [ -0.0000,  -0.0000,   0.0000],
        [ -0.0000,  -0.0000,   0.0000],
        [ -0.0000,  -0.0000,   0.0000],
        [ -0.0000,  -0.0000,   0.0000],
        [ -0.0000,  -0.0000,   0.0000],
        [ -0.0000,  -0.0000,   0.0000],
        [ -0.0000,  -0.0000,   0.0000]], device='cuda:0')

In [32]:
uncropped_data['metadata']

{'protein_name': ['1C3P_A', '1C3P_A', '1C52_A'],
 'gt_atom37_positions': [array([[[11.4989996 , 40.52899933, 35.87400055],
          [12.47000027, 40.72499847, 34.89699936],
          [13.83100033, 40.78699875, 35.56999969],
          ...,
          [ 0.        ,  0.        ,  0.        ],
          [ 0.        ,  0.        ,  0.        ],
          [ 0.        ,  0.        ,  0.        ]],
  
         [[14.88399982, 40.1590004 , 35.02999878],
          [16.27499962, 40.41199875, 35.3409996 ],
          [17.03000069, 40.67599869, 34.03699875],
          ...,
          [ 0.        ,  0.        ,  0.        ],
          [16.74099922, 37.24900055, 40.73300171],
          [ 0.        ,  0.        ,  0.        ]],
  
         [[17.06999969, 42.00699997, 33.76399994],
          [17.87999916, 42.48600006, 32.5929985 ],
          [19.37400055, 42.38399887, 32.66799927],
          ...,
          [ 0.        ,  0.        ,  0.        ],
          [14.49699974, 47.25099945, 31.4829998 ],
        

In [3]:
import torch.nn as nn
from openfold.model.structure_module import StructureModule
from openfold.model.mmc.model import MMCRefinementModel
from openfold.model.mmc.core import load_structure_auxillary_modules
from openfold.config import model_config
from openfold.model.model import AlphaFold
from openfold.utils.import_weights import import_jax_weights_

In [12]:
from run_refinement_model import 

In [4]:
# import_jax_weights_(model, jax_param_path, version="model_3")
# jax_param_path
sm, am, ev = load_structure_auxillary_modules(return_evoformer=True)

Model version: model_3
Successfully loaded JAX parameters at openfold/resources/params/params_model_3.npz


In [5]:
# Initialize the MMC refinement model
refinement_model = MMCRefinementModel(
    structure_module=sm,
    aux_heads=am,
    c_z=128,
    c_s=384,
    c_hidden_mul=128,
    c_hidden_att=32,
    no_heads_pair=4,
    no_heads_single=4,
    transition_n=4,
    dropout_rate=0.2,
    num_cycles=3,
)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move models to device
structure_module = sm.to(device)
refinement_model = refinement_model.to(device)

# Print model summary
print(f"Structure Module Parameters: {sum(p.numel() for p in structure_module.parameters())}")
print(f"Refinement Model Parameters: {sum(p.numel() for p in refinement_model.parameters())}")


Using device: cuda
Structure Module Parameters: 2019116
Refinement Model Parameters: 3376986


In [ ]:
final_evoformer_block = ev.blocks[-1]

# Get the pair stack from the final evoformer block
evo_pair_stack = final_evoformer_block.pair_stack

# Copy weights from triangle attention modules
refinement_model.pair_refinement_module.tri_att_start.load_state_dict(
    evo_pair_stack.tri_att_start.state_dict(), strict=False
)
refinement_model.pair_refinement_module.tri_att_end.load_state_dict(
    evo_pair_stack.tri_att_end.state_dict(), strict=False
)

In [16]:
final_evoformer_block = ev.blocks[-1]
# Get the pair stack from the final evoformer block
evo_pair_stack = final_evoformer_block.pair_stack

# Copy weights from triangle attention modules
refinement_model.pair_refinement_module.tri_att_start.load_state_dict(
    evo_pair_stack.tri_att_start.state_dict(), strict=False
)
refinement_model.pair_refinement_module.tri_att_end.load_state_dict(
    evo_pair_stack.tri_att_end.state_dict(), strict=False
)

print("Successfully initialized triangle attention modules")

Successfully initialized triangle attention modules


In [15]:
# cpu_device = torch.device("cpu")
# result = tensor_tree_map(lambda x: x.to(cpu_device), result)
# torch.cuda.empty_cache()
torch.cuda.memory_summary()

'|===========================================================================|\n|                  PyTorch CUDA memory summary, device ID 0                 |\n|---------------------------------------------------------------------------|\n|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |\n|===========================================================================|\n|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |\n|---------------------------------------------------------------------------|\n| Allocated memory      |  23016 KiB | 362946 KiB | 377674 KiB | 354658 KiB |\n|       from large pool |   8352 KiB |   8352 KiB |   8352 KiB |      0 KiB |\n|       from small pool |  14664 KiB | 359778 KiB | 369322 KiB | 354658 KiB |\n|---------------------------------------------------------------------------|\n| Active memory         |  23016 KiB | 362946 KiB | 377674 KiB | 354658 KiB |\n|       from large pool |   8352 KiB |   8352 KiB |

# 5/6. Prepare Batch for Model Input + Run Forward


In [16]:
# Set models to evaluation mode for testing
structure_module.eval()
refinement_model.train()

loss_fn = RefinementLoss()

debugging = True

# Run forward pass with gradient tracking
with torch.autograd.set_detect_anomaly(debugging):
    with torch.set_grad_enabled(True):
        # Make sure inputs require gradients

        if "metadata" in batch:
            batch.pop("metadata")
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        batch = tensor_tree_map(lambda x: x.to(device), batch)
        output = refinement_model(
            batch['pair'],
            batch['single'],
            batch['feats'],
            pair_mask=batch['pair_mask'],
            seq_mask=batch['seq_mask'],
            external_grad=batch['forces'],
        )

        total_loss, breakdown = loss_fn(output, batch, _return_breakdown=True)

        total_loss.backward()

        cpu_device = torch.device("cpu")
        output = tensor_tree_map(lambda x: x.to(cpu_device), output)
        batch = tensor_tree_map(lambda x: x.to(cpu_device), batch)
    #     torch.cuda.empty_cache()

    #     print("Forward pass completed successfully!")
    #     print("Output keys:", list(output.keys()))

    #     # Print shapes of key outputs
    #     for key, value in output.items():
    #         if isinstance(value, torch.Tensor):
    #             print(f"  {key}: Tensor with shape {value.shape}")
    #         elif isinstance(value, list) and len(value) > 0 and isinstance(value[0], torch.Tensor):
    #             print(f"  {key}: List of {len(value)} tensors, first with shape {value[0].shape}")

In [18]:
batch[]

dict_keys(['pair', 'single', 'pair_mask', 'seq_mask', 'all_atom_positions', 'all_atom_mask', 'aatype', 'atom14_atom_exists', 'residx_atom14_to_atom37', 'residx_atom37_to_atom14', 'atom37_atom_exists', 'atom14_gt_exists', 'atom14_gt_positions', 'atom14_alt_gt_positions', 'atom14_alt_gt_exists', 'atom14_atom_is_ambiguous', 'feats', 'seq_length', 'forces', 'residue_index', 'alt_naming_is_better', 'renamed_atom14_gt_positions', 'renamed_atom14_gt_exists', 'pseudo_beta', 'pseudo_beta_mask', 'rigidgroups_gt_frames', 'rigidgroups_gt_exists', 'rigidgroups_group_exists', 'rigidgroups_group_is_ambiguous', 'rigidgroups_alt_gt_frames', 'backbone_rigid_tensor', 'backbone_rigid_mask', 'resolution', 'torsion_angles_sin_cos', 'alt_torsion_angles_sin_cos', 'torsion_angles_mask', 'chi_angles_sin_cos', 'chi_mask'])

In [20]:
from openfold.model.mmc.metrics import calculate_ca_rmsd, calculate_atom14_rmsd
output_positions = output['final_atom_positions']
gt_positions = batch['atom14_gt_positions']
atom14_atom_exists = batch['atom14_atom_exists']
rmsd = calculate_ca_rmsd(output_positions, gt_positions, atom14_atom_exists)

In [21]:
print(output.keys(), batch.keys())
print(output['sm'].keys())
# batch['residue_index'] = batch['feats']['residue_index']

dict_keys(['pair', 'single', 'positions', 'cycle_positions', 'cycle_pair_deltas', 'cycle_single_deltas', 'acceptance_rates', 'sm', 'lddt_logits', 'plddt', 'distogram_logits', 'experimentally_resolved_logits']) dict_keys(['pair', 'single', 'pair_mask', 'seq_mask', 'all_atom_positions', 'all_atom_mask', 'aatype', 'atom14_atom_exists', 'residx_atom14_to_atom37', 'residx_atom37_to_atom14', 'atom37_atom_exists', 'atom14_gt_exists', 'atom14_gt_positions', 'atom14_alt_gt_positions', 'atom14_alt_gt_exists', 'atom14_atom_is_ambiguous', 'feats', 'seq_length', 'forces', 'residue_index'])
dict_keys(['frames', 'sidechain_frames', 'unnormalized_angles', 'angles', 'positions', 'states', 'single'])


# Test Distributed Data Sampler

In [40]:
from openfold.model.mmc.data import *
# Path to the test predictions directory
project_root = "/gpfs/data/rsingh47/hp_protein_folding/protein_folding"
predictions_dir = os.path.join(project_root, "output_mmc", "test_predictions")
print(f"Predictions directory: {predictions_dir}")

datasets = build_dataset(predictions_dir, pH='5')

Predictions directory: /gpfs/data/rsingh47/hp_protein_folding/protein_folding/output_mmc/test_predictions


In [41]:
dataloaders = create_data_loaders(datasets, batch_size=8)

Batch size: 8


In [19]:
for i, batch in enumerate(dataloaders['train']):
    print("hi")
    print(batch['seq_length'], "hi")

In [43]:
batch = next(iter(dataloaders['train']))

In [44]:
batch['seq_length']

tensor([305, 285,  51, 228, 285, 323, 177, 228])